In [2]:
import os
import sys

# Tự động cấu hình Java và Hadoop (winutils) cho Spark trên Windows
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot"
os.environ["HADOOP_HOME"] = r"D:\NYC_Taxi_Prj\hadoop"
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.path.join(os.environ["HADOOP_HOME"], "bin")

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("nyc-taxi")
    .getOrCreate()
)


In [3]:
df = spark.read.parquet(
    "data/raw/yellow_tripdata_2026-01.parquet"
)

In [4]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [5]:
df.show(4)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|                 N|         239|    

In [6]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 3724889
Columns: 20


In [7]:
df.describe().show()

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+------------------+-------------------+---------------------+------------------+--------------------+-------------------+-------------------+
|summary|          VendorID|   passenger_count|    trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|     DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|       tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee| cbd_congestion_fee|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+------------------+-------------------+---------------------+----

In [8]:
df2 = (
    df

    .withColumn(
        "trip_duration_min",

        (
            unix_timestamp(
                "tpep_dropoff_datetime"
            )

            -

            unix_timestamp(
                "tpep_pickup_datetime"
            )

        )/60
    )# Lấy thời gian trả khách - thời gian nhận đón khách

    .withColumn(
        "tip_ratio",

        col("tip_amount")
        /

        col("fare_amount")
    )

    .withColumn(
        "pickup_date",

        to_date(
            "tpep_pickup_datetime"
        )
    )
)

In [9]:
df2.select(
"trip_duration_min",
"tip_ratio",
"pickup_date"
).show(10)

+------------------+-------------------+-----------+
| trip_duration_min|          tip_ratio|pickup_date|
+------------------+-------------------+-----------+
|              5.55| 0.5083333333333333| 2026-01-01|
| 5.716666666666667|                0.0| 2026-01-01|
| 8.883333333333333|0.23364485981308414| 2026-01-01|
|              42.8| 0.2870801033591731| 2026-01-01|
|              13.5| 0.2851851851851852| 2026-01-01|
|              13.6| 0.3514084507042254| 2026-01-01|
|10.633333333333333|                0.0| 2026-01-01|
|24.616666666666667|               0.25| 2026-01-01|
|37.733333333333334|0.23083109919571046| 2026-01-01|
| 9.583333333333334| 0.2205607476635514| 2026-01-01|
+------------------+-------------------+-----------+
only showing top 10 rows


In [10]:
from pyspark.sql.functions import *

df_different= df.select(
(
col("fare_amount")
+
col("extra")
+
col("mta_tax")
+
col("tip_amount")
+
col("tolls_amount")
+
col("improvement_surcharge")
+
col("congestion_surcharge")
+
col("Airport_fee")
+
col("cbd_congestion_fee")

-

col("total_amount")

).alias("diff")

)

In [11]:
from pyspark.sql.functions import *

check_df = df.withColumn(

"diff",

col("fare_amount")
+
col("extra")
+
col("mta_tax")
+
col("tip_amount")
+
col("tolls_amount")
+
col("improvement_surcharge")
+
col("congestion_surcharge")
+
col("Airport_fee")
+
col("cbd_congestion_fee")

-

col("total_amount")
)

In [12]:
check_df.select (

"fare_amount"
,
"extra"
,
"mta_tax"
,
"tip_amount"
,
"tolls_amount"
,
"improvement_surcharge"
,
"congestion_surcharge"
,
"Airport_fee"
,
"cbd_congestion_fee"
,"diff"
).where(
    col("diff").isNotNull()
    ).orderBy(
    col("diff").asc()
).show(10)

+-----------+-----+-------+----------+------------+---------------------+--------------------+-----------+------------------+------------------+
|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|congestion_surcharge|Airport_fee|cbd_congestion_fee|              diff|
+-----------+-----+-------+----------+------------+---------------------+--------------------+-----------+------------------+------------------+
|       70.0|  0.0|    0.5|     17.79|        7.46|                  1.0|                 2.5|       1.75|              0.75|-5.000000000000014|
|       70.0|  0.0|    0.5|     17.54|        6.94|                  1.0|                 2.5|       1.75|               0.0|-5.000000000000014|
|       70.0|  0.0|    0.5|     17.44|        7.46|                  1.0|                 2.5|        0.0|              0.75|-5.000000000000014|
|       70.0|  0.0|    0.5|     17.54|        6.94|                  1.0|                 2.5|       1.75|               0.0|-5.00

In [13]:
from pyspark.sql.functions import col, sum

# Tính tổng số dòng Null ở mỗi cột
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|        1088058|            0|   1088058|           1088058|           0|    

In [14]:
df.filter(col("trip_distance") > 100).select("trip_distance", "fare_amount", "total_amount").show(10)


+-------------+-----------+------------+
|trip_distance|fare_amount|total_amount|
+-------------+-----------+------------+
|     36610.27|     105.27|      117.21|
|       100.37|       12.8|       21.06|
|       112.49|      689.7|      722.39|
|       123.18|      797.5|      807.19|
|       121.69|      762.5|      799.19|
|        122.2|      400.0|      415.24|
|       107.16|      438.4|      560.74|
|       116.06|      380.0|       381.0|
|        192.8|       33.5|        39.6|
|       106.06|      660.3|      672.01|
+-------------+-----------+------------+
only showing top 10 rows


In [15]:
df.groupBy("passenger_count").count().orderBy("passenger_count").show()


+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|           NULL|1088058|
|              0|  14787|
|              1|2150994|
|              2| 334370|
|              3|  72864|
|              4|  49738|
|              5|   9184|
|              6|   4887|
|              7|      2|
|              8|      4|
|              9|      1|
+---------------+-------+



In [18]:
from pyspark.sql.functions import col, sum, round

total_rows = df.count()
null_counts = df.select([
    round((sum(col(c).isNull().cast("int")) / total_rows * 100), 2).alias(c) 
    for c in df.columns
])
null_counts.show()
print("Chuyến đi có khoảng cách = 0:", df.filter(col("trip_distance") <= 0).count())
print("Chuyến đi có khoảng cách > 100 dặm:", df.filter(col("trip_distance") > 100).count())



+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|     0.0|                 0.0|                  0.0|          29.21|          0.0|     29.21|             29.21|         0.0|    

In [19]:
print("Số chuyến đi bị âm tiền (total_amount < 0):", df.filter(col("total_amount") < 0).count())
print("Số chuyến đi miễn phí (total_amount = 0):", df.filter(col("total_amount") == 0).count())
print("Giá trị giao dịch lớn nhất (max total_amount):", df.selectExpr("max(total_amount)").collect()[0][0])


Số chuyến đi bị âm tiền (total_amount < 0): 39984
Số chuyến đi miễn phí (total_amount = 0): 433
Giá trị giao dịch lớn nhất (max total_amount): 2560.2


In [ ]:
# Check dropoff trước pickup
invalid_time = df.filter(col("tpep_dropoff_datetime") <= col("tpep_pickup_datetime")).count()
print("Số chuyến có thời gian trả khách trước hoặc bằng thời gian đón:", invalid_time)

Số chuyến có thời gian trả khách trước hoặc bằng thời gian đón: 45070


: 